# XAI Adapter Quickstart

A short walkthrough for instantiating XAI methods through `src.xai_adapter` and rendering the results as tables.

Sections:
1. LOFO — dependency-free smoke test
2. Native SHAP — `ShapTreeExplainer` (sklearn RF) and `ShapLinearExplainer`
3. KernelSHAP — model-agnostic `shap.KernelExplainer`
4. CSV-backed precomputed explanations
5. Surrogate methods (rules + weights) from raw data
6. CoXAM assets loader

In [1]:
from pathlib import Path
import sys

# Support running from repo root or from the tutorials/ subfolder
repo_root = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from src.data_loaders import UnifiedDataLoader, XAIDatasetParser
from src.xai_adapter import (
    ShapDeepExplainer,
    ShapGradientExplainer,
    ShapLinearExplainer,
    ShapTreeExplainer,
    create_coxam_xai_method,
    create_xai_method,
)

assets_root = repo_root / "assets"
print("repo_root:", repo_root)

repo_root: /Users/wangzhuoyulucas/Documents/GitHub/xaik-tool-cognitive-agent


## 1 · LOFO — Leave-One-Feature-Out

`lofo` requires no extra libraries — it only needs a `predict_fn` and background data. Good as a smoke-test for the `.fit()` / `.explain()` API.

In [2]:
def predict_proba(X):
    X = np.asarray(X, dtype=float)
    p1 = np.clip(0.2 + 0.35 * X[:, 0] + 0.45 * X[:, 1] - 0.10 * X[:, 2], 0.0, 1.0)
    return np.column_stack([1.0 - p1, p1])

X_train = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
])
X_test = np.array([[1.0, 0.5, 0.0]])
FEATURE_NAMES = ["v0", "v1", "v2"]

lofo = create_xai_method("lofo", predict_fn=predict_proba).fit(X_train)
lofo_result = lofo.explain(X_test)

pd.DataFrame(lofo_result.values, columns=FEATURE_NAMES)

,v0,v1,v2
0,0.175,0.0,0.05


## 2 · Native SHAP Wrappers

The adapter ships four wrappers that forward directly to the optimised SHAP explainers — no sampling overhead:

| Class | Registry key | Backed by | Best for |
|---|---|---|---|
| `ShapTreeExplainer` | `shap_tree` | `shap.TreeExplainer` | XGBoost / LightGBM / sklearn trees |
| `ShapLinearExplainer` | `shap_linear` | `shap.LinearExplainer` | Linear / logistic regression |
| `ShapDeepExplainer` | `shap_deep` | `shap.DeepExplainer` | PyTorch / TF networks |
| `ShapGradientExplainer` | `shap_gradient` | `shap.GradientExplainer` | PyTorch / TF (gradient-based) |

All four expose the same `fit(X) → explain(instances) → XAIAdapterResult` contract as every other adapter.

### 2a · ShapTreeExplainer — sklearn RandomForest

In [3]:
try:
    from sklearn.ensemble import RandomForestClassifier
    import shap

    rng = np.random.default_rng(0)
    X_rf = rng.standard_normal((120, 3))
    y_rf = (X_rf[:, 0] + X_rf[:, 1] > 0).astype(int)

    rf = RandomForestClassifier(n_estimators=50, random_state=0).fit(X_rf, y_rf)

    tree_exp = ShapTreeExplainer(
        model=rf,
        background_data=X_rf,   # used for interventional feature_perturbation
        target=1,
    )
    # ShapTreeExplainer fits itself in __init__; no separate .fit() call needed
    tree_result = tree_exp.explain(X_rf[:5])

    display(pd.DataFrame(
        tree_result.values,
        columns=FEATURE_NAMES,
    ).assign(base_value=tree_result.base_values))

    # Can also be created via the registry:
    # tree_exp = create_xai_method("shap_tree", model=rf, background_data=X_rf, target=1)

except ImportError as e:
    print(f"Skipped — missing dependency: {e}")

/Users/wangzhuoyulucas/anaconda3/envs/rlnb_ibl_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,v0,v1,v2,base_value
0,-0.108700,-0.099800,-0.112500,0.441
1,-0.129867,-0.209267,0.018133,0.441
2,0.289133,0.237833,0.032033,0.441
3,-0.282733,-0.134333,-0.023933,0.441
4,-0.330133,-0.092633,0.021767,0.441


### 2b · ShapLinearExplainer — sklearn LogisticRegression

In [4]:
try:
    from sklearn.linear_model import LogisticRegression
    import shap  # noqa: F811

    rng = np.random.default_rng(1)
    X_lr = rng.standard_normal((100, 3))
    y_lr = (X_lr[:, 0] + 0.5 * X_lr[:, 1] > 0).astype(int)

    lr = LogisticRegression(random_state=0).fit(X_lr, y_lr)

    linear_exp = ShapLinearExplainer(
        model=lr,
        background_data=X_lr,
        nsamples=100,
        target=1,
    )
    linear_result = linear_exp.explain(X_lr[:5])

    display(pd.DataFrame(
        linear_result.values,
        columns=FEATURE_NAMES,
    ).assign(base_value=linear_result.base_values))

except ImportError as e:
    print(f"Skipped — missing dependency: {e}")

,v0,v1,v2,base_value
0,0.957769,1.638525,0.032617,0.307357
1,-4.875884,1.793849,0.038418,0.307357
2,-2.164866,1.192424,0.034325,0.307357
3,0.775721,0.167234,0.043438,0.307357
4,-2.870749,-0.187666,-0.008038,0.307357


### 2c · ShapDeepExplainer / ShapGradientExplainer — PyTorch

These require `torch` and `shap`. The cell is guarded so the notebook still runs without them.

In [5]:
try:
    import torch
    import torch.nn as nn
    import shap  # noqa: F811

    rng = np.random.default_rng(2)
    X_nn = rng.standard_normal((80, 3)).astype(np.float32)
    y_nn = (X_nn[:, 0] + X_nn[:, 1] > 0).astype(int)

    net = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-2)
    Xt = torch.tensor(X_nn)
    yt = torch.tensor(y_nn)
    for _ in range(200):
        optimizer.zero_grad()
        nn.CrossEntropyLoss()(net(Xt), yt).backward()
        optimizer.step()
    net.eval()

    bg = torch.tensor(X_nn[:20])  # small background for deep / gradient explainer

    deep_exp = ShapDeepExplainer(model=net, background_data=bg, target=1)
    deep_result = deep_exp.explain(X_nn[:4])
    print("DeepExplainer SHAP values:")
    display(pd.DataFrame(deep_result.values, columns=FEATURE_NAMES))

    grad_exp = ShapGradientExplainer(model=net, background_data=bg, target=1)
    grad_result = grad_exp.explain(X_nn[:4])
    print("GradientExplainer SHAP values:")
    display(pd.DataFrame(grad_result.values, columns=FEATURE_NAMES))

except ImportError as e:
    print(f"Skipped — missing dependency: {e}")

AttributeError: 'numpy.ndarray' object has no attribute 'detach'

## 3 · KernelSHAP — Model-Agnostic

`KernelShap` (registry key `shap_kernel` / `shap`) wraps `shap.KernelExplainer`. It works with any `predict_fn` but is slower than TreeExplainer.

In [6]:
try:
    import shap  # noqa: F811

    kernel_shap = create_xai_method(
        "shap_kernel",
        predict_fn=predict_proba,
        background_data=X_train,
        n_background_samples=4,
        target=1,
    )
    ks_result = kernel_shap.explain(X_test)
    pd.DataFrame(ks_result.values, columns=FEATURE_NAMES)

except ImportError as e:
    print(f"Skipped — missing dependency: {e}")

100%|██████████| 1/1 [00:00<00:00, 203.58it/s]


## 4 · CSV-Backed Precomputed Explanations

Use the CSV adapter when instances, AI predictions, and explanation vectors already come from an external pipeline.

In [7]:
csv_df = pd.DataFrame([
    {"instanceId": 0, "pred": 1, "v0": 0.2, "v1": 0.7, "v2": 0.1, "a0_i": 0.4, "a1_i": 0.5, "a2_i": 0.1},
    {"instanceId": 1, "pred": 0, "v0": 0.8, "v1": 0.1, "v2": 0.3, "a0_i": 0.6, "a1_i": 0.2, "a2_i": 0.2},
])

csv_dataset = XAIDatasetParser.from_dataframe(csv_df)
csv_method = create_xai_method("csv", dataset=csv_dataset)
records = csv_dataset.get_records([0, 1])

pd.DataFrame([
    {
        "instanceId": record.instance_id,
        "ai_prediction": record.ai_prediction,
        "features": record.features.tolist(),
        "explanation": record.explanation.tolist(),
    }
    for record in records
])

,instanceId,ai_prediction,features,explanation
0,0.0,1.0,"[0.2, 0.7, 0.1]","[0.4, 0.5, 0.1]"
1,1.0,0.0,"[0.8, 0.1, 0.3]","[0.6, 0.2, 0.2]"


## 5 · Surrogate Methods From Raw Data

When a CSV has instances and AI predictions but no precomputed CoXAM tables, fit `rules` (decision tree) and `weights` (logistic regression) directly from `X, y`.

In [8]:
surrogate_df = pd.DataFrame([
    {"instanceId": 0, "pred": 0, "v0": 0.0, "v1": 0.0, "v2": 0.0},
    {"instanceId": 1, "pred": 0, "v0": 0.2, "v1": 0.1, "v2": 0.0},
    {"instanceId": 2, "pred": 1, "v0": 0.9, "v1": 0.8, "v2": 1.0},
    {"instanceId": 3, "pred": 1, "v0": 1.0, "v1": 0.9, "v2": 1.0},
])

X_surr = surrogate_df[["v0", "v1", "v2"]].to_numpy()
y_surr = surrogate_df["pred"].to_numpy()

rules = create_xai_method("rules", app_id="toy", model_name="external", depth=2).fit(X_surr, y_surr)
weights = create_xai_method("weights", app_id="toy", model_name="external", variant="sparse", top_k=2).fit(X_surr, y_surr)

display(pd.DataFrame(rules.explain(X_surr).values).add_prefix("rule_path_v"))
display(pd.DataFrame(weights.explain(X_surr).values).add_prefix("weight_contribution_v"))

,rule_path_v0,rule_path_v1,rule_path_v2
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,0.0,1.0,0.0
3,0.0,1.0,0.0


,weight_contribution_v0,weight_contribution_v1,weight_contribution_v2
0,0.0,0.000000,0.000000
1,0.0,0.061569,0.000000
2,0.0,0.492549,0.627168
3,0.0,0.554117,0.627168


## 6 · CoXAM Assets Loader

Use `create_coxam_xai_method(...)` when explanations live in `assets/explanations/coxam` and should be loaded through `UnifiedDataLoader`.

In [9]:
try:
    coxam_loader = UnifiedDataLoader.from_assets(source="coxam", assets_root=str(assets_root))
    instance_ids = [0, 1, 2]
    X_coxam = coxam_loader.get_features(instance_ids, normalize=False)

    rules_cx = create_coxam_xai_method(
        coxam_loader,
        method_type="rules",
        app_id="wine_quality",
        model_name="mlp",
        depth=3,
    )
    weights_cx = create_coxam_xai_method(
        coxam_loader,
        method_type="weights",
        app_id="wine_quality",
        model_name="mlp",
        variant="sparse",
    )

    rules_result = rules_cx.explain(X_coxam)
    weights_result = weights_cx.explain(X_coxam)

    display(pd.DataFrame({
        "instanceId": instance_ids,
        "rules_prediction": [item["class_index"] for item in rules_result.metadata["predictions"]],
        "weights_probability": weights_result.metadata["predictions"],
    }))
    display(pd.DataFrame(weights_result.values).add_prefix("weight_contribution_v"))
except Exception as exc:
    pd.DataFrame([{"status": "CoXAM assets example skipped", "reason": f"{type(exc).__name__}: {exc}"}])

,instanceId,rules_prediction,weights_probability
0,0,0,1.0
1,1,1,1.0
2,2,1,1.0


,weight_contribution_v0,weight_contribution_v1,weight_contribution_v2,weight_contribution_v3,weight_contribution_v4,weight_contribution_v5,weight_contribution_v6,weight_contribution_v7,weight_contribution_v8
0,46.154596,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,58.975317,0.000000,0.0,-23.597429,0.0,0.0,0.0,0.0,0.0
2,65.385678,3.246654,0.0,-17.698072,0.0,0.0,0.0,0.0,0.0
